In [1]:
%%writefile src/fairness.py
"""Fairness check: predictive equality = similar false positive rates for under 50 and 50 and over."""
import numpy as np
import pandas as pd

from src import evaluation


def group_report(y, scores, groups, thresholds):
    """thresholds can be one number or a dict {group: threshold}."""
    y = np.asarray(y)
    scores = np.asarray(scores)
    groups = np.asarray(groups)
    rows = []
    for g in np.unique(groups):
        m = groups == g
        thr = thresholds[g] if isinstance(thresholds, dict) else thresholds
        r = evaluation.rates_at_threshold(y[m], scores[m], thr)
        rows.append({"group": g, "applications": int(m.sum()), "fraud_rate": float(y[m].mean()),
                     "TPR": r["TPR"], "FPR": r["FPR"], "flagged_share": r["flagged_share"],
                     "precision": r["precision"]})
    return pd.DataFrame(rows)


def predictive_equality_ratio(report):
    """smallest group FPR divided by the largest group FPR (1.0 = perfectly equal)."""
    fprs = report["FPR"]
    if len(fprs) < 2 or fprs.max() == 0:
        return float("nan")
    return float(fprs.min() / fprs.max())


def group_thresholds(y, scores, groups, fpr=0.05):
    """Mitigation: one threshold per group so every group has the same false positive rate (set on validation data)."""
    y = np.asarray(y)
    scores = np.asarray(scores)
    groups = np.asarray(groups)
    return {g: evaluation.threshold_at_fpr(y[groups == g], scores[groups == g], fpr) for g in np.unique(groups)}

Writing src/fairness.py


In [2]:
import config
from src.data import age_group, load_baf, prepare, split_by_month
from src.evaluation import threshold_at_fpr
from src.fairness import group_report, group_thresholds, predictive_equality_ratio
from src.modelling import fit_model

df = load_baf(config.DATA_DIR / "Base.csv")
train, valid, _ = split_by_month(df)

sample_tr = train.head(5000)
scorer = fit_model("logistic_regression", prepare(sample_tr), sample_tr[config.TARGET].values)

sample_va = valid.head(5000)
y_va = sample_va[config.TARGET].values
s_va = scorer.score(prepare(sample_va))
groups_va = age_group(sample_va[config.AGE].values)

base_thr = threshold_at_fpr(y_va, s_va, config.FPR_HIGH)
rep_base = group_report(y_va, s_va, groups_va, base_thr)
print("Shared threshold report:")
print(rep_base)
print("Predictive equality ratio (shared):", round(predictive_equality_ratio(rep_base), 3))

mit_thrs = group_thresholds(y_va, s_va, groups_va, fpr=config.FPR_HIGH)
rep_mit = group_report(y_va, s_va, groups_va, mit_thrs)
print("\nGroup-specific threshold report:")
print(rep_mit)
print("Predictive equality ratio (mitigated):", round(predictive_equality_ratio(rep_mit), 3))

Shared threshold report:
         group  applications  fraud_rate       TPR       FPR  flagged_share  \
0  50 and over           881    0.020431  0.444444  0.085747       0.093076   
1     Under 50          4119    0.007526  0.290323  0.042564       0.044428   

   precision  
0   0.097561  
1   0.049180  
Predictive equality ratio (shared): 0.496

Group-specific threshold report:
         group  applications  fraud_rate       TPR       FPR  flagged_share  \
0  50 and over           881    0.020431  0.333333  0.050985       0.056754   
1     Under 50          4119    0.007526  0.322581  0.050147       0.052197   

   precision  
0   0.120000  
1   0.046512  
Predictive equality ratio (mitigated): 0.984
